# 0. Optional Steps

The steps in this section are optional and used for debug logging

## 0.1 Set the environment to use a proxy

In [ ]:
import os

# Setup proxy through local proxy; bypass proxy for metadata endpoint to avoid any issues with Microsoft authentication libraries
os.environ["HTTP_PROXY"] = "http://127.0.0.1:8000"
os.environ["HTTPS_PROXY"] = "http://127.0.0.1:8000"
os.environ["NO_PROXY"] = "169.254.169.254"

# Add private CA root cert to trusted cert bundle for SDKs that use requests and httpx
os.environ["REQUESTS_CA_BUNDLE"] = "/Users/matthewfelton/Library/Preferences/httptoolkit/ca.pem"
os.environ["SSL_CERT_FILE"] = "/Users/matthewfelton/Library/Preferences/httptoolkit/ca.pem"

import os

# Setup proxy through local proxy; bypass proxy for metadata endpoint to avoid any issues with Microsoft authentication libraries
os.environ["HTTP_PROXY"] = "http://127.0.0.1:8000"
os.environ["HTTPS_PROXY"] = "http://127.0.0.1:8000"
os.environ["NO_PROXY"] = "169.254.169.254"

# Add private CA root cert to trusted cert bundle for SDKs that use requests and httpx
os.environ["REQUESTS_CA_BUNDLE"] = "/Users/matthewfelton/Library/Preferences/httptoolkit/ca.pem"
os.environ["SSL_CERT_FILE"] = "/Users/matthewfelton/Library/Preferences/httptoolkit/ca.pem"

# 1. Setup the notebook

The steps in this section setup the notebook

## 1.1 Install required libraries

In [ ]:
%pip install -r requirements.txt

## 1.2 Setup a logger

In [ ]:
# Setup a Python logger
import logging
import sys

def configure_logging(level="ERROR"):
    """This function configures logging for code being run based on the specified level.
    Args:
        level (str): The logging level to use (e.g., "DEBUG", "INFO", "WARNING", "ERROR", "CRITICAL").
    """

    try:
        # Convert the level string to uppercase so it matches what the logging library expects
        logging_level = getattr(logging, level.upper(), None)

        # Setup a logging format
        logging.basicConfig(
            level=logging_level,
            format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
            handlers=[logging.StreamHandler(sys.stdout)]
        )
    except Exception as e:
        print(f"Failed to set up logging: {e}", file=sys.stderr)
        sys.exit(1) 

configure_logging("ERROR")

## 1.3 Load environment variables

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(".env")

FOUNDRY_MODEL_OPENAI_ENDPOINT = os.getenv("FOUNDRY_MODEL_OPENAI_ENDPOINT")
FOUNDRY_MODEL_APIM_ENDPOINT = os.getenv("FOUNDRY_MODEL_APIM_ENDPOINT")
FOUNDRY_MODEL_ACCOUNT_ENDPOINT = os.getenv("FOUNDRY_MODEL_ACCOUNT_ENDPOINT")
FOUNDRY_MODEL_PROJECT_ENDPOINT = os.getenv("FOUNDRY_MODEL_PROJECT_ENDPOINT")
ENTRA_TENANT_ID = os.getenv("ENTRA_TENANT_ID")
BU1_ENTRA_SERVICE_PRINCIPAL_CLIENT_ID = os.getenv("BU1_ENTRA_SERVICE_PRINCIPAL_CLIENT_ID")
BU1_ENTRA_SERVICE_PRINCIPAL_CLIENT_SECRET = os.getenv("BU1_ENTRA_SERVICE_PRINCIPAL_CLIENT_SECRET")
BU2_ENTRA_SERVICE_PRINCIPAL_CLIENT_ID = os.getenv("BU2_ENTRA_SERVICE_PRINCIPAL_CLIENT_ID")
BU2_ENTRA_SERVICE_PRINCIPAL_CLIENT_SECRET = os.getenv("BU2_ENTRA_SERVICE_PRINCIPAL_CLIENT_SECRET")
LLM_DEPLOYMENT_NAME = os.getenv("LLM_DEPLOYMENT_NAME")


# 2. Multi-turn using Responses API server-side storage of responses

This section uses the OpenAI Responses API. Responses are [stored server-side using the property **store** set to **True**](https://developers.openai.com/api/reference/resources/responses/methods/create).

In [ ]:
# Get an Entra token

from azure.identity import ClientSecretCredential, get_bearer_token_provider

credential = ClientSecretCredential(
    client_id=BU1_ENTRA_SERVICE_PRINCIPAL_CLIENT_ID,
    client_secret=BU1_ENTRA_SERVICE_PRINCIPAL_CLIENT_SECRET,
    tenant_id=ENTRA_TENANT_ID
)

token_provider = get_bearer_token_provider(
    credential,
    "https://ai.azure.com/.default"
)
print(token_provider())


## 2.1 Send to Foundry openai.azure.com endpoint

In [ ]:
from openai import OpenAI, NotFoundError
import json

# Create the Azure OpenAI Service client
# 
client = OpenAI(
    base_url=FOUNDRY_MODEL_OPENAI_ENDPOINT,
    api_key=token_provider
)

first_question = "What is the capital of Hawaii?"

first_response = client.responses.create(
    model=LLM_DEPLOYMENT_NAME,
    # Store responses server-side
    store=True,
    instructions="You are a helpful assistant.",
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": first_question
                }
            ]
        }
    ]
)

# Use the prior response id as context for a follow up question
follow_up_question = "What is the population?"

follow_up_response = client.responses.create(
    model=LLM_DEPLOYMENT_NAME,
    store=True,
    instructions="You are a helpful assistant.",
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": follow_up_question
                }
            ]
        }
    ],
    previous_response_id=first_response.id
)

print(f"User: {first_question}")
print(f"Assistant: {first_response.output_text}")
print(f"User: {follow_up_question}")
print(f"Assistant: {follow_up_response.output_text}")
print(f"--------------------------------------------")

# Fetch the response id to check if it is cached
print(f"Fetching response ID: {first_response.id}...")
try:
    first_response_cached = client.responses.retrieve(first_response.id)
    print("Responses are being cached")
except NotFoundError:
    print("Responses are not being cached")
print(f"--------------------------------------------")
# Dump the raw JSON of the responses for debugging purposes
print("Response API call response:")
print(first_response.model_dump_json(indent=2))

## 2.2 Send to Foundry account-level endpoint (services.ai.azure.com)

In [ ]:
from openai import OpenAI, NotFoundError
import json

# Create the Azure OpenAI Service client
# 
client = OpenAI(
    base_url=FOUNDRY_MODEL_ACCOUNT_ENDPOINT,
    api_key=token_provider
)

first_question = "What is the capital of New Hampshire?"

first_response = client.responses.create(
    model=LLM_DEPLOYMENT_NAME,
    # Store responses server-side
    store=True,
    instructions="You are a helpful assistant.",
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": first_question
                }
            ]
        }
    ]
)

# Use the prior response id as context for a follow up question
follow_up_question = "What is the population?"

follow_up_response = client.responses.create(
    model=LLM_DEPLOYMENT_NAME,
    # Store responses server-side
    store=True,
    instructions="You are a helpful assistant.",
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": follow_up_question
                }
            ]
        }
    ],
    previous_response_id=first_response.id
)

print(f"User: {first_question}")
print(f"Assistant: {first_response.output_text}")
print(f"User: {follow_up_question}")
print(f"Assistant: {follow_up_response.output_text}")
print(f"--------------------------------------------")

# Fetch the response id to check if it is cached
print(f"Fetching response ID: {first_response.id}...")
try:
    first_response_cached = client.responses.retrieve(first_response.id)
    print("Responses are being cached")
except NotFoundError:
    print("Responses are not being cached")
print(f"--------------------------------------------")
# Dump the raw JSON of the responses for debugging purposes
print("Response API call response:")
print(first_response.model_dump_json(indent=2))

## 2.3 Send to Foundry project-level endpoint (services.ai.azure.com/api/project)

In [ ]:
from openai import OpenAI, NotFoundError
import json

# Create the Azure OpenAI Service client
# 
client = OpenAI(
    base_url=FOUNDRY_MODEL_PROJECT_ENDPOINT,
    api_key=token_provider
)

first_question = "What is the capital of New Hampshire?"

first_response = client.responses.create(
    model=LLM_DEPLOYMENT_NAME,
    # Store responses server-side
    store=True,
    instructions="You are a helpful assistant.",
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": first_question
                }
            ]
        }
    ]
)

# Use the prior response id as context for a follow up question
follow_up_question = "What is the population?"

follow_up_response = client.responses.create(
    model=LLM_DEPLOYMENT_NAME,
    # Store responses server-side
    store=True,
    instructions="You are a helpful assistant.",
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": follow_up_question
                }
            ]
        }
    ],
    previous_response_id=first_response.id
)

print(f"User: {first_question}")
print(f"Assistant: {first_response.output_text}")
print(f"User: {follow_up_question}")
print(f"Assistant: {follow_up_response.output_text}")
print(f"--------------------------------------------")

# Fetch the response id to validate it was not saved
print(f"Fetching response ID: {first_response.id}...")
try:
    first_response_cached = client.responses.retrieve(first_response.id)
    print("Responses are being cached")
except NotFoundError:
    print("Responses are not being cached")
print(f"--------------------------------------------")
# Dump the raw JSON of the responses for debugging purposes
print("Response API call response:")
print(first_response.model_dump_json(indent=2))

## 2.4 Send to Foundry AI Gateway endpoint

In [ ]:
from openai import OpenAI, NotFoundError, DefaultHttpxClient
from http.cookies import SimpleCookie
import json

received_cookie_values = set()
cookie_checks = []

def report_cookie_exchange(response):
    sent_cookies = SimpleCookie()
    sent_cookies.load(response.request.headers.get("cookie", ""))
    reused_names = sorted(
        name for name, cookie in sent_cookies.items()
        if (name, cookie.value) in received_cookie_values
    )
    received_names = set()
    for header in response.headers.get_list("set-cookie"):
        response_cookies = SimpleCookie()
        response_cookies.load(header)
        for name, cookie in response_cookies.items():
            received_names.add(name)
            received_cookie_values.add((name, cookie.value))
    cookie_checks.append({
        "received": bool(response.headers.get_list("set-cookie")),
        "reused": bool(reused_names)
    })
    print(f"\nCookie check: {response.request.method} {response.request.url.path} ({response.status_code})")
    print(f"  Sent cookie names: {sorted(sent_cookies) or 'none'}")
    print(f"  Received Set-Cookie names: {sorted(received_names) or 'none'}")
    print(f"  Previously received cookies sent back with matching values: {reused_names or 'none'}")

# Create the Azure OpenAI Service client
# 
client = OpenAI(
    base_url=FOUNDRY_MODEL_APIM_ENDPOINT,
    api_key=token_provider,
    http_client=DefaultHttpxClient(event_hooks={"response": [report_cookie_exchange]})
)

first_question = "What is the capital of New Hampshire?"

first_response = client.responses.create(
    model=LLM_DEPLOYMENT_NAME,
    # Store responses server-side
    store=True,
    instructions="You are a helpful assistant.",
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": first_question
                }
            ]
        }
    ]
)

# Use the prior response id as context for a follow up question
follow_up_question = "What is the population?"

follow_up_response = client.responses.create(
    model=LLM_DEPLOYMENT_NAME,
    # Store responses server-side
    store=True,
    instructions="You are a helpful assistant.",
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": follow_up_question
                }
            ]
        }
    ],
    previous_response_id=first_response.id
)

flower_question = "What is that state's official flower?"
flower_response = client.responses.create(
    model=LLM_DEPLOYMENT_NAME,
    store=True,
    instructions="You are a helpful assistant.",
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": flower_question
                }
            ]
        }
    ],
    previous_response_id=follow_up_response.id
)

flag_question = "What are the colors of that state's flag?"
flag_response = client.responses.create(
    model=LLM_DEPLOYMENT_NAME,
    store=True,
    instructions="You are a helpful assistant.",
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": flag_question
                }
            ]
        }
    ],
    previous_response_id=flower_response.id
)

print(f"User: {first_question}")
print(f"Assistant: {first_response.output_text}")
print(f"User: {follow_up_question}")
print(f"Assistant: {follow_up_response.output_text}")
print(f"User: {flower_question}")
print(f"Assistant: {flower_response.output_text}")
print(f"User: {flag_question}")
print(f"Assistant: {flag_response.output_text}")
print(f"--------------------------------------------")

# Fetch the response id to check if it is cached
print(f"Fetching response ID: {first_response.id}...")
try:
    first_response_cached = client.responses.retrieve(first_response.id)
    print("Responses are being cached")
except NotFoundError:
    print("Responses are not being cached")
print(f"--------------------------------------------")
print("Cookie received:", any(check["received"] for check in cookie_checks))
print("Cookie sent back with matching value:", any(check["reused"] for check in cookie_checks))
# Dump the raw JSON of the responses for debugging purposes
print("Response API call response:")
print(first_response.model_dump_json(indent=2))

# 3. Multi-turn using Responses API and Conversations API

This section uses the OpenAI Responses API. [Conversations](https://developers.openai.com/api/reference/resources/conversations/methods/create) API to store user's conversation across API calls. Using the conversations API requires that the store property be set to True. Note that this will also store responses.

Microsoft Foundry only supports the conversations API at the project endpoint.


In [ ]:
# Get an Entra token

from azure.identity import ClientSecretCredential, get_bearer_token_provider

credential = ClientSecretCredential(
    client_id=BU1_ENTRA_SERVICE_PRINCIPAL_CLIENT_ID,
    client_secret=BU1_ENTRA_SERVICE_PRINCIPAL_CLIENT_SECRET,
    tenant_id=ENTRA_TENANT_ID
)

token_provider = get_bearer_token_provider(
    credential,
    "https://ai.azure.com/.default"
)
print(token_provider())


In [ ]:
from openai import OpenAI, NotFoundError, DefaultHttpxClient
from http.cookies import SimpleCookie
import json

received_cookie_values = set()
cookie_checks = []

def report_cookie_exchange(response):
    sent_cookies = SimpleCookie()
    sent_cookies.load(response.request.headers.get("cookie", ""))
    reused_names = sorted(
        name for name, cookie in sent_cookies.items()
        if (name, cookie.value) in received_cookie_values
    )
    received_names = set()
    for header in response.headers.get_list("set-cookie"):
        response_cookies = SimpleCookie()
        response_cookies.load(header)
        for name, cookie in response_cookies.items():
            received_names.add(name)
            received_cookie_values.add((name, cookie.value))
    cookie_checks.append({
        "received": bool(response.headers.get_list("set-cookie")),
        "reused": bool(reused_names)
    })
    print(f"\nCookie check: {response.request.method} {response.request.url.path} ({response.status_code})")
    print(f"  Sent cookie names: {sorted(sent_cookies) or 'none'}")
    print(f"  Received Set-Cookie names: {sorted(received_names) or 'none'}")
    print(f"  Previously received cookies sent back with matching values: {reused_names or 'none'}")

# Create the Azure OpenAI Service client
# 
client = OpenAI(
    base_url=FOUNDRY_MODEL_PROJECT_ENDPOINT,
    api_key=token_provider,
    http_client=DefaultHttpxClient(event_hooks={"response": [report_cookie_exchange]})
)

# Create a conversation object
conversation_id = client.conversations.create().id

# Spin up the first question
first_question = "What is the capital of Maine?"
first_response = client.responses.create(
    model=LLM_DEPLOYMENT_NAME,
    store=True,
    instructions="You are a helpful assistant.",
    input=[
        {
            "type": "message",
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": first_question
                }
            ]
        }
    ],
    conversation=conversation_id
)
print("Conversation after first turn:")
print(client.conversations.retrieve(conversation_id).model_dump_json(indent=2))
# List the items actually stored in the conversation to confirm turns are persisted
print("Conversation items after first turn:")
print(client.conversations.items.list(conversation_id).model_dump_json(indent=2))

# Spin up a follow-up question and use the conversation as context
follow_up_question = "What is the population?"
follow_up_response = client.responses.create(
    model=LLM_DEPLOYMENT_NAME,
    store=True,
    instructions="You are a helpful assistant.",
    input=[
        {
            "type": "message",
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": follow_up_question
                }
            ]
        }
    ],
    conversation=conversation_id
)
print("Conversation items after follow-up turn:")
print(client.conversations.items.list(conversation_id).model_dump_json(indent=2))

print(f"User: {first_question}")
print(f"Assistant: {first_response.output_text}")
print(f"User: {follow_up_question}")
print(f"Assistant: {follow_up_response.output_text}")
print(f"--------------------------------------------")

# Fetch the response id to validate it was not saved
print(f"Fetching response ID: {first_response.id}...")
try:
    first_response_cached = client.responses.retrieve(first_response.id)
    print("Responses are being cached")
except NotFoundError:
    print("Responses are not being cached")
print(f"--------------------------------------------")
print("Cookie received:", any(check["received"] for check in cookie_checks))
print("Cookie sent back with matching value:", any(check["reused"] for check in cookie_checks))
# Dump the raw JSON of the conversation
print("Response API call response:")
print(client.conversations.retrieve(conversation_id).model_dump_json(indent=2))


# 4. Multi-Turn with Responses API and a tool

This section uses the OpenAI Responses API. The model is provided with the Microsoft Learn tool to use for additional research. Responses are not [stored server-side by setting the property **store** set to **False**](https://developers.openai.com/api/reference/resources/responses/methods/create).

In [ ]:
# Get an Entra token

from azure.identity import ClientSecretCredential, get_bearer_token_provider

credential = ClientSecretCredential(
    client_id=BU1_ENTRA_SERVICE_PRINCIPAL_CLIENT_ID,
    client_secret=BU1_ENTRA_SERVICE_PRINCIPAL_CLIENT_SECRET,
    tenant_id=ENTRA_TENANT_ID
)

token_provider = get_bearer_token_provider(
    credential,
    "https://ai.azure.com/.default"
)
print(token_provider())


In [ ]:
from openai import OpenAI, DefaultHttpxClient
from http.cookies import SimpleCookie
import textwrap

received_cookie_values = set()
cookie_checks = []

def report_cookie_exchange(response):
    sent_cookies = SimpleCookie()
    sent_cookies.load(response.request.headers.get("cookie", ""))
    reused_names = sorted(
        name for name, cookie in sent_cookies.items()
        if (name, cookie.value) in received_cookie_values
    )
    received_names = set()
    for header in response.headers.get_list("set-cookie"):
        response_cookies = SimpleCookie()
        response_cookies.load(header)
        for name, cookie in response_cookies.items():
            received_names.add(name)
            received_cookie_values.add((name, cookie.value))
    cookie_checks.append({
        "received": bool(response.headers.get_list("set-cookie")),
        "reused": bool(reused_names)
    })
    print(f"\nCookie check: {response.request.method} {response.request.url.path} ({response.status_code})")
    print(f"  Sent cookie names: {sorted(sent_cookies) or 'none'}")
    print(f"  Received Set-Cookie names: {sorted(received_names) or 'none'}")
    print(f"  Previously received cookies sent back with matching values: {reused_names or 'none'}")

# Create the Azure OpenAI Service client
# 
client = OpenAI(
    base_url=FOUNDRY_MODEL_PROJECT_ENDPOINT,
    api_key=token_provider,
    http_client=DefaultHttpxClient(event_hooks={"response": [report_cookie_exchange]})
)

first_response = client.responses.create(
    # Store the response in the service so it can be referenced in a follow up
    store=True,
    model=LLM_DEPLOYMENT_NAME,
    # Set the instruction
    instructions="""You are an expert Microsoft Azure assistant.

            Instructions:
            - Use the learn tool to search Microsoft Learn documentation before answering
            - Provide accurate, well-grounded responses based on official documentation
            - If the learn tool is unavailable or returns no results, clearly state: "I cannot answer this question at this time as I'm unable to access the documentation"
            - Keep responses concise and relevant to Azure
            - At the end of the response include citations with links back to the Learn documentation given the responses you got from the tool. For example: "Sources: [1](link), [2](link)"

            If a question is outside the scope of Microsoft Azure, politely redirect the user.""",
    # Configure the Microsoft Learn MCP Server tool
    tools=[
        {
            "type": "mcp",
            "server_label": "mslearn",
            "server_description": "Microsoft Learn documentation MCP server",
            "server_url": "https://learn.microsoft.com/api/mcp",
            "require_approval": "never"
        }
    ],
    # Force the model to use the learn tool
    tool_choice={
        "type": "mcp",
        "server_label": "mslearn"
    },
    # If the model supports reasoning, print it out
    reasoning={
        "effort": "medium", 
        "summary": "detailed"
    },
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": "I have a TB of data that needs to be accessed by external users. What type of Azure storage should I use?"
                }
            ]
        }
    ]
)

# response.reasoning only echoes the requested config; the actual reasoning text is in output items of type "reasoning"
reasoning_items = [item for item in first_response.output if item.type == "reasoning"]
if reasoning_items:
    print("\n=== Reasoning Summary ===")
    for item in reasoning_items:
        for segment in item.summary:
            print(textwrap.fill(segment.text, width=100))
    print("\n=== Effort Level ===")
    print(first_response.reasoning.effort)

# Tool activity (e.g. MCP tool discovery/invocation) shows up as its own output item types
tool_items = [item for item in first_response.output if item.type in ("mcp_list_tools", "mcp_call", "mcp_approval_request")]
if tool_items:
    print("\n=== Tool Calls ===")
    for item in tool_items:
        print(item.model_dump_json(indent=2))

print("\n=== Response Text ===")
print(first_response.output_text)
response_id = first_response.id
print(f"--------------------------------------------")
print("Cookie received:", any(check["received"] for check in cookie_checks))
print("Cookie sent back with matching value:", any(check["reused"] for check in cookie_checks))
